In [1]:
import mlflow
import pandas as pd 
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score , precision_score , recall_score , f1_score
import re
import string
import nltk
from nltk.corpus import stopwords
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.stem import WordNetLemmatizer
import numpy as np


c:\Users\Lenovo\anaconda3\envs\atlas\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
df = pd.read_csv(
    r"C:\practice\Text_Classification_ML_pipeline\notebooks\data.csv"
)
df.head()

,review,sentiment
0,Every great gangster movie has under-currents ...,positive
1,"I just saw this film last night, and I have to...",positive
2,This film is mildly entertaining if one neglec...,negative
3,Quentin Tarantino's partner in crime Roger Ava...,negative
4,I sat through this on TV hoping because of the...,negative


In [3]:
def lemmatization(text):
    lemmatizer = WordNetLemmatizer()
    text = str(text).split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    stop_words = set(stopwords.words('english'))
    text = str(text).split()
    text = [word for word in text if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    text = ''.join([char for char in str(text) if not char.isdigit()])
    text = ' '.join(text.split())
    return text

def lower_case(text):
    text = str(text).split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_urls(text):
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'',text)

def removing_punctuations(text):
    text = re.sub('[%s]'%re.escape(string.punctuation),' ',text)
    text = text.replace('؛', "")
    text = re.sub(r'\s+',' ',text).strip()
    return text

def normalize_text(df):
    try:
        df['review']= df['review'].apply(lower_case)
        df['review']= df['review'].apply(remove_stop_words)
        df['review']= df['review'].apply(removing_numbers)
        df['review']= df['review'].apply(removing_punctuations)
        df['review']= df['review'].apply(removing_urls)
        df['review']= df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise


In [4]:
df = normalize_text(df)
df.head()

,review,sentiment
0,every great gangster movie under current human...,positive
1,saw film last night say loved every minute tak...,positive
2,film mildly entertaining one neglect acknowled...,negative
3,quentin tarantino s partner crime roger avary ...,negative
4,sat tv hoping name would worth time but dear g...,negative


In [5]:
df['sentiment'].value_counts()

sentiment
negative    269
positive    231
Name: count, dtype: int64

In [6]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [7]:
df['sentiment']=df['sentiment'].map({'positive':1,'negative':0})
df.head()

,review,sentiment
0,every great gangster movie under current human...,1
1,saw film last night say loved every minute tak...,1
2,film mildly entertaining one neglect acknowled...,0
3,quentin tarantino s partner crime roger avary ...,0
4,sat tv hoping name would worth time but dear g...,0


In [8]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [9]:
vectorizer = CountVectorizer(max_features=100)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [10]:
X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.25 , random_state=42)

In [12]:
import dagshub
mlflow.set_tracking_uri('https://dagshub.com/Shrutichauha7/Text_Classification_ML_pipeline.mlflow')
dagshub.init(repo_owner='Shrutichauha7', repo_name='Text_Classification_ML_pipeline', mlflow=True)

mlflow.set_experiment("Logistic Regression Baseline")

Accessing as Shrutichauha7

Initialized MLflow to track repo "Shrutichauha7/Text_Classification_ML_pipeline"

Repository Shrutichauha7/Text_Classification_ML_pipeline initialized!

2026/06/13 22:30:50 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/092113e99904410586fd03367eb1be8a', creation_time=1781370053643, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1781370053643, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, trace_location=None, workspace='default'>

In [13]:
import mlflow
import logging 
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score , precision_score , recall_score , f1_score

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()

    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer","bag of words")
        mlflow.log_param("num_features",100)
        mlflow.log_param("test_size",0.25)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)

        logging.info("Fitting the model...")
        model.fit(X_train,y_train)
        logging.info("Model Training complete")

        logging.info("Logging model Parameters...")
        mlflow.log_param("model","Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test,y_pred)
        precision = precision_score(y_test,y_pred)
        recall = recall_score(y_test,y_pred)
        f1 = f1_score(y_test,y_pred)


        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy",accuracy)
        mlflow.log_metric("precision",precision)
        mlflow.log_metric("recall",recall)
        mlflow.log_metric("f1_score",f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model,"model")

        end_time = time.time()
        logging.info("Model training nad logging completed")


        logging.info(f"Accuracy:{accuracy}")
        logging.info(f"Precision:{precision}")
        logging.info(f"Recall:{recall}")
        logging.info(f"F1 Score:{f1}")

    except Exception as e:
        logging.error(f"An error occured: {e}",exc_info=True)






2026-06-13 23:01:48,899 - INFO - Starting MLflow run...
2026-06-13 23:01:55,826 - INFO - Logging preprocessing parameters...
2026-06-13 23:02:00,361 - INFO - Initializing Logistic Regression model...
2026-06-13 23:02:00,361 - INFO - Fitting the model...
2026-06-13 23:02:00,476 - INFO - Model Training complete
2026-06-13 23:02:00,476 - INFO - Logging model Parameters...
2026-06-13 23:02:00,842 - INFO - Making predictions...
2026-06-13 23:02:00,842 - INFO - Calculating evaluation metrics...
2026-06-13 23:02:00,869 - INFO - Logging evaluation metrics...
2026-06-13 23:02:02,378 - INFO - Saving and logging the model...
2026/06/13 23:02:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/13 23:02:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The reco

🏃 View run honorable-deer-813 at: https://dagshub.com/Shrutichauha7/Text_Classification_ML_pipeline.mlflow/#/experiments/0/runs/3e4d1fa2ff0540f5913ce51ebd170c2d
🧪 View experiment at: https://dagshub.com/Shrutichauha7/Text_Classification_ML_pipeline.mlflow/#/experiments/0
